# Combine multiple OCEAN LoRAs

Each entry in `CONFIGS` is a dict mapping OCEAN slug (`o_plus`, `n_minus`, ...) to a scale.
For every config we load the named adapters from the canonical `OCEAN_REGISTRY` onto Llama-3.1-8B-Instruct, activate them all simultaneously with the requested scales (via `load_and_scale_adapters`), and generate responses to a small sample of open-ended OCEAN questions from `data/ocean_open_ended/`.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src_dev.common.lora_catalogue import OCEAN_REGISTRY
from src_dev.utils.lora_composition import WeightedAdapter, load_and_scale_adapters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Top-of-notebook config
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
QUESTIONS_DIR = Path("../data/ocean_open_ended")
N_QUESTIONS_PER_TRAIT = 3
MAX_NEW_TOKENS = 200

# Each config is a dict[slug, scale]; all listed adapters are activated together.
CONFIGS: list[dict[str, float]] = [
    {"o_plus": 0.0, "c_plus": 0.0, "e_plus": 0.0, "a_plus": 0.0, "n_plus": 0.0},
    {"o_plus": 0.1, "c_plus": 0.1, "e_plus": 0.1, "a_plus": 0.1, "n_plus": 0.1},
    {"o_plus": 0.2, "c_plus": 0.2, "e_plus": 0.2, "a_plus": 0.2, "n_plus": 0.2},
    {"o_plus": 0.3, "c_plus": 0.3, "e_plus": 0.3, "a_plus": 0.3, "n_plus": 0.3},
    {"o_plus": 0.4, "c_plus": 0.4, "e_plus": 0.4, "a_plus": 0.4, "n_plus": 0.4},
    {"o_plus": 0.5, "c_plus": 0.5, "e_plus": 0.5, "a_plus": 0.5, "n_plus": 0.5},
]

In [ ]:
def load_questions(questions_dir: Path, n_per_trait: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    records: list[dict] = []
    for path in sorted(questions_dir.glob("*.jsonl")):
        with path.open() as f:
            rows = [json.loads(line) for line in f if line.strip()]
        sampled = rng.sample(rows, k=min(n_per_trait, len(rows)))
        records.extend(sampled)
    return records

questions = load_questions(QUESTIONS_DIR, N_QUESTIONS_PER_TRAIT, SEED)
print(f"Loaded {len(questions)} questions across {len(set(q['trait'] for q in questions))} traits")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
)
model.eval()
print(f"Loaded base model on {next(model.parameters()).device}")

In [ ]:
@torch.inference_mode()
def generate(model, tokenizer, question: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    messages = [{"role": "user", "content": question}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output_ids[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
# load_and_scale_adapters wraps `model` in a PeftModel on first call. Between
# configs we delete the adapters and peel the PEFT wrapper off so the next
# config starts from a clean base.
rows: list[dict] = []
current_model = model

for cfg_idx, scale_map in enumerate(CONFIGS):
    print(f"\n=== Config {cfg_idx}: {scale_map} ===")
    adapters = [
        WeightedAdapter(path=OCEAN_REGISTRY[slug].adapter_ref, scale=float(scale))
        for slug, scale in scale_map.items()
    ]
    peft_model, adapter_names, _ = load_and_scale_adapters(
        current_model,
        adapters=adapters,
        adapter_name_prefix=f"cfg{cfg_idx}",
    )
    peft_model.eval()

    for q in questions:
        response = generate(peft_model, tokenizer, q["question"])
        rows.append({
            "config_idx": cfg_idx,
            "config": scale_map,
            "trait": q["trait"],
            "facet": q["facet"],
            "question": q["question"],
            "response": response,
        })

    for name in adapter_names:
        peft_model.delete_adapter(name)
    current_model = peft_model.get_base_model()

df = pd.DataFrame(rows)
df

In [ ]:
pd.set_option("display.max_colwidth", None)
for idx, group in df.groupby("config_idx"):
    cfg = group["config"].iloc[0]
    print(f"\n========== Config {idx}: {cfg} ==========")
    for _, row in group.iterrows():
        print(f"\n[{row['trait']} / {row['facet']}] {row['question']}")
        print(f"  -> {row['response']}")